In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import random

# 1. Load Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 2. Dataset Augmentation
datagen = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1)

# 3. Model Builder Function
def build_model(optimizer_name='adam', dropout_rate=0.2, l2_val=0.01):
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_val)),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(l2_val)),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ])

    # Map string names to actual Keras optimizers
    optimizers = {
        'adagrad': tf.keras.optimizers.Adagrad(),
        'rmsprop': tf.keras.optimizers.RMSprop(),
        'adam': tf.keras.optimizers.Adam()
    }

    model.compile(optimizer=optimizers[optimizer_name],
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 4. Manual "Random Search" (Quick & No extra libraries)
best_acc = 0
best_params = {}
options = {
    'optimizer': ['adagrad', 'rmsprop', 'adam'],
    'dropout': [0.2, 0.5],
    'l2': [0.001, 0.01]
}

print("Starting Hyperparameter Tuning...")
for i in range(3):  # Running 3 random combinations for speed
    opt = random.choice(options['optimizer'])
    drp = random.choice(options['dropout'])
    reg = random.choice(options['l2'])

    print(f"Trial {i+1}: Testing {opt}, Dropout: {drp}, L2: {reg}")
    model = build_model(opt, drp, reg)

    # Early Stopping
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2)

    # Train briefly to find the best setup
    model.fit(datagen.flow(x_train, y_train, batch_size=64),
              epochs=2, validation_data=(x_test, y_test),
              callbacks=[early_stop], verbose=0)

    _, acc = model.evaluate(x_test, y_test, verbose=0)
    if acc > best_acc:
        best_acc = acc
        best_params = {'optimizer': opt, 'dropout': drp, 'l2': reg}

# 5. Final Training with Best Params
print(f"\nBest Results: {best_params} with {best_acc*100:.2f}% accuracy")
final_model = build_model(best_params['optimizer'], best_params['dropout'], best_params['l2'])
final_model.fit(datagen.flow(x_train, y_train, batch_size=32), epochs=5, validation_data=(x_test, y_test))


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Starting Hyperparameter Tuning...
Trial 1: Testing adagrad, Dropout: 0.2, L2: 0.01


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Trial 2: Testing adagrad, Dropout: 0.2, L2: 0.01
Trial 3: Testing rmsprop, Dropout: 0.2, L2: 0.001

Best Results: {'optimizer': 'rmsprop', 'dropout': 0.2, 'l2': 0.001} with 96.17% accuracy
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 32s 17ms/step - accuracy: 0.6750 - loss: 1.2235 - val_accuracy: 0.9482 - val_loss: 0.3700
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - accuracy: 0.8792 - loss: 0.5812 - val_accuracy: 0.9573 - val_loss: 0.3024
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - accuracy: 0.8916 - loss: 0.5041 - val_accuracy: 0.9615 - val_loss: 0.2693
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 30s 16ms/step - accuracy: 0.8969 - loss: 0.4756 - val_accuracy: 0.9625 - val_loss: 0.2593
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 31s 16ms/step - accuracy: 0.8996 - loss: 0.4652 - val_accuracy: 0.9563 - val_loss: 0.2649
